# 📚 | Import Libraries

In [ ]:
import random
import wandb
import torch
import sklearn
import numpy as np
import pandas as pd
import polars as pl
import os
import plotly.express as px
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from datasets import Dataset
from scipy.special import softmax

from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA (NVIDIA GPU)")
else:
    device = torch.device("cpu")
    print("Using CPU")

print(f"Device: {device}")

## Library version

# ⚙️ | Configuration

In [ ]:
class CFG:
    INPUT_DIR = "/kaggle/input/llm-classification-finetuning"
    OUTPUT_DIR = "/kaggle/working"
    MODEL_DIR = "/kaggle/working/models"
    BASE_MODEL_PATH = "/kaggle/input/deberta-v3-base/deberta-v3-base"
    
    lr = 5e-5   # learning rate
    batch_size = 8  # Training batch size
    epochs = 1  # Training epochs
    seed = 42  # Random seed
    weight_decay = 0.1
    label2name = {0: 'winner_model_a', 1: 'winner_model_b', 2: 'winner_tie'}
    name2label = {v: k for k, v in label2name.items()}
    class_labels = list(label2name.keys())
    class_names = list(label2name.values())
    max_length = 768
    
    lora_rank = 4
    lora_alpha = 16
    lora_dropout = 0.0
    lora_modules = ['q_proj', 'v_proj']

In [ ]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.seed)

In [ ]:
os.environ["WANDB_PROJECT"] = "llm-classification-kaggle"
os.environ["WANDB_LOG_MODEL"] = "end"
os.environ["WANDB_WATCH"] = "false"

os.makedirs(CFG.OUTPUT_DIR, exist_ok=True)
os.makedirs(CFG.MODEL_DIR, exist_ok=True)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# 📖 | Meta Data

The competition dataset comprises user interactions from the ChatBot Arena. In each interaction, a judge presents one or more prompts to two different large language models and then indicates which model provided the more satisfactory response. The training data contains `55,000` rows, with an expected `25,000` rows in the test set.

## Files
### `train.csv`
- `id`: Unique identifiers for each row.
- `model_[a/b]`: Model identity, present in train.csv but not in test.csv.
- `prompt`: Input prompt given to both models.
- `response_[a/b]`: Model_[a/b]'s response to the prompt.
- `winner_model_[a/b/tie]`: Binary columnns indicating the judge's selection (ground truth target)

### `test.csv`
- `id`: Unique identifier for each row.
- `prompt`: Input prompt given to both models.
- `response_[a/b]`: Model_[a/b]'s response to the prompt.

## Get Data

In [ ]:
# Load Train Data
df_train = pl.read_csv(f"{CFG.INPUT_DIR}/train.csv")
df_test = pl.read_csv(f"{CFG.INPUT_DIR}/test.csv")
df_sample = pl.read_csv(f"{CFG.INPUT_DIR}/sample_submission.csv")

In [ ]:
print(df_train["prompt"][0])

In [ ]:
df_train.head()

## Clean Data

In [ ]:
# Process prompt and response_[a/b] in csv from list-str to str
def process_text(input_str):
    '''input_str = "[""sent_1","sent_2","..."]" '''
    input_str = input_str.strip('[]')
    sentences = [s.strip('"') for s in input_str.split('","')]
    
    # handle "null" by replacing "null" to empty string
    sentences = [s if s!="null" else "" for s in sentences]

    return ' '.join(sentences)

In [ ]:
df_train = df_train.with_columns([
    pl.col("prompt").map_elements(process_text,return_dtype=str).alias("prompt"),
    pl.col("response_a").map_elements(process_text,return_dtype=str).alias("response_a"),
    pl.col("response_b").map_elements(process_text,return_dtype=str).alias("response_b"),
])

df_test = df_test.with_columns([
    pl.col("prompt").map_elements(process_text,return_dtype=str).alias("prompt"),
    pl.col("response_a").map_elements(process_text,return_dtype=str).alias("response_a"),
    pl.col("response_b").map_elements(process_text,return_dtype=str).alias("response_b"),
])

In [ ]:
df_train = df_train.with_columns([
    pl.when(pl.col("winner_model_a") == 1)
    .then(pl.lit(CFG.name2label["winner_model_a"]))
    .when(pl.col("winner_model_b") == 1)
    .then(pl.lit(CFG.name2label["winner_model_b"]))
    .when(pl.col("winner_tie") == 1)
    .then(pl.lit(CFG.name2label["winner_tie"]))
    .otherwise(pl.lit(None))
    .alias("label"),

    # Add label name column
    pl.when(pl.col("winner_model_a") == 1)
      .then(pl.lit("winner_model_a"))
      .when(pl.col("winner_model_b") == 1)
      .then(pl.lit("winner_model_b"))
      .when(pl.col("winner_tie") == 1)
      .then(pl.lit("winner_tie"))
      .otherwise(pl.lit(None))
      .alias("class_name")
])

In [ ]:
df_train

# 🎨 | Exploratory Data Analysis (EDA)

## Large Language Model Distribution

In [ ]:
# Ép về pandas trước
model_a = df_train["model_a"].to_pandas()
model_b = df_train["model_b"].to_pandas()

llm_df = pd.concat([model_a, model_b])
counts = llm_df.value_counts().reset_index()
counts.columns = ['LLM', 'Count']

fig = px.bar(counts, x='LLM', y='Count',
             title='Distribution of LLMs',
             color='Count', color_continuous_scale='viridis')

fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Winning Distribution

In [ ]:
class_name = df_train['class_name'].to_pandas()
counts = class_name.value_counts().reset_index()
counts.columns = ['Winner', 'Win Count']

fig = px.bar(counts, x='Winner', y='Win Count',
             title='Winner distribution for Train Data',
             labels={'Winner': 'Winner', 'Win Count': 'Win Count'},
             color='Winner', color_continuous_scale='viridis')

fig.update_layout(xaxis_title="Winner", yaxis_title="Win Count")
fig.show()

# 🍽️ | Preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.BASE_MODEL_PATH)
sep = tokenizer.sep_token

In [ ]:
def apply_fmt(df, sep_token):
    return df.with_columns(
        (
            pl.lit("Prompt: ") + pl.col("prompt") + pl.lit(f" {sep_token} ") +
            pl.lit("Response A: ") + pl.col("response_a") + pl.lit(f" {sep_token} ") +
            pl.lit("Response B: ") + pl.col("response_b")
        ).alias("text")
    )

In [ ]:
df_train = apply_fmt(df_train, sep)
df_test = apply_fmt(df_test, sep)

In [ ]:
df_train

In [ ]:
train_texts, valid_texts, train_labels, valid_labels = train_test_split(
    df_train["text"].to_list(), 
    df_train["label"].to_list(), 
    test_size=0.1, 
    random_state=CFG.seed)

In [ ]:
train_texts[0]

In [ ]:
class LLMDataset(torch.utils.data.Dataset):
    def __init__(self, texts, tokenizer, labels=None, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {key: val.squeeze() for key, val in encoding.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])

        return item

    def __len__(self):
        return len(self.texts)

In [ ]:
train_dataset = LLMDataset(list(train_texts), tokenizer, list(train_labels))
valid_dataset = LLMDataset(list(valid_texts), tokenizer, list(valid_labels))
test_dataset = LLMDataset(list(df_test["text"]), tokenizer)

# 🤖 | Modeling

In [ ]:
base_model = AutoModelForSequenceClassification.from_pretrained(
    CFG.BASE_MODEL_PATH, 
    num_labels=len(CFG.class_labels),
)

# PiSSA init
lora_config = LoraConfig(
    init_lora_weights='pissa_niter_8',
    r=CFG.lora_rank,
    lora_alpha=CFG.lora_alpha,
    lora_dropout=CFG.lora_dropout,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

peft_model = get_peft_model(base_model, lora_config)
peft_model.print_trainable_parameters()

# 📏 | Metric

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

# 🚂 | Training

In [ ]:
os.environ["WANDB_PROJECT"] = "kaggle-llm-classification"
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"
os.environ["WANDB_MODE"] = "disabled"

steps_per_epoch = len(train_dataset)// CFG.batch_size
logging_steps = max(steps_per_epoch // 10, 1)

train_args = TrainingArguments(
    output_dir=f"{CFG.MODEL_DIR}/results",
    learning_rate=CFG.lr,                      # 0.00001
    per_device_train_batch_size=CFG.batch_size,  # batch size per device during training
    per_device_eval_batch_size=CFG.batch_size,   # batch size for evaluation
    num_train_epochs=CFG.epochs,
    weight_decay=CFG.weight_decay,
    eval_strategy="epoch",
    save_strategy="no",
    metric_for_best_model="accuracy",
    disable_tqdm=False,
    logging_steps=logging_steps,
    logging_first_step=True,
    report_to=[],
    dataloader_num_workers=0
)

peft_model.to(device)

trainer = Trainer(
    model=peft_model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

wandb.finish()

In [ ]:
final_model_path = f"{CFG.MODEL_DIR}/final"

trainer.save_model(final_model_path)
tokenizer.save_pretrained(final_model_path)

# 📝 | Submission

In [ ]:
def load_model(model_path):
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, 
        num_labels=len(CFG.class_labels))
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    return model, tokenizer

infer_model, infer_tokenizer = load_model(final_model_path)

test_trainer = Trainer(
    model=infer_model,
    args=train_args,
    train_dataset=train_dataset,
    eval_dataset=valid_dataset,
    compute_metrics=compute_metrics
)

# Predict test dataset
predictions = test_trainer.predict(test_dataset)
pred_logits = predictions.predictions
pred_probs = softmax(pred_logits, axis=1)

# Create submission dataframe
submission = pd.DataFrame({
    "id": df_test["id"],
    "winner_model_a": pred_probs[:, 0],
    "winner_model_b": pred_probs[:, 1],
    "winner_model_tie": pred_probs[:, 2],
})

submission.to_csv("submission.csv", index=False)
submission.head()